# Notebook 2 - Regresi Prediksi IPK Mahasiswa
**Mata Kuliah:** Pembelajaran Mesin - UAS Genap T.A. 2025/2026
**Program Studi:** Informatika, Universitas Atma Jaya Yogyakarta
**Dosen:** Yohanes Sigit Purnomo, Ph.D. & Theresia Devi Indriasari, Ph.D.

---
Model regresi untuk memprediksi IPK kumulatif mahasiswa berdasarkan gaya belajar, kebiasaan belajar, lingkungan, teknologi, dan faktor akademik.

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
import math
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.preprocessing    import StandardScaler, OneHotEncoder
from sklearn.impute           import SimpleImputer

from sklearn.feature_selection import (SelectKBest, SelectPercentile,
                                        f_regression, mutual_info_regression)

from sklearn.model_selection  import (train_test_split, GridSearchCV,
                                       KFold, cross_validate, cross_val_score)

from sklearn.linear_model     import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble         import (RandomForestRegressor,
                                       GradientBoostingRegressor,
                                       ExtraTreesRegressor)
from sklearn.svm              import SVR

from sklearn.metrics          import (r2_score, mean_absolute_error,
                                       mean_squared_error)

import joblib
import sklearn

RANDOM_STATE = 99
print(f"scikit-learn   = {sklearn.__version__}")
print(f"joblib         = {joblib.__version__}")
print(f"pandas         = {pd.__version__}")
print(f"numpy          = {np.__version__}")

scikit-learn   = 1.9.0
joblib         = 1.5.3
pandas         = 3.0.4
numpy          = 2.2.6


## 2. Data Loading
Dataset yang digunakan adalah hasil survei preferensi belajar, kebiasaan belajar, dan faktor akademik mahasiswa yang telah **dianonimkan**.


In [2]:
FILE_PATH = 'ANONIM_Survei Preferensi Belajar, Kebiasaan Belajar, dan Faktor Akademik Mahasiswa (Responses).xlsx'
df_raw = pd.read_excel(FILE_PATH)

print(f"Shape: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")

Shape: 324 baris x 89 kolom


In [3]:
df_raw.head(3)

,Timestamp,Pernyataan Persetujuan,Program studi,Semester saat ini,Jenis kelamin,Usia (tahun),Status tempat tinggal selama kuliah,Apakah Anda bekerja sambil kuliah?,Sumber pembiayaan utama kuliah saya saat ini adalah:,Rata-rata waktu belajar mandiri per hari di luar jam kuliah,...,"Saat belajar di rumah, saya mengerjakan latihan soal, tugas, atau kuis untuk memahami materi.","Saat belajar sendiri, saya mencoba langsung langkah-langkah atau prosedur yang sedang dipelajari.","Saat mempelajari materi, saya menggunakan contoh kasus nyata agar lebih mudah memahami konsep.","Saat belajar di rumah, saya melakukan simulasi, praktik, eksperimen, atau demonstrasi kecil sesuai materi yang dipelajari.","Saat merasa belum paham, saya mencoba sendiri sampai mengetahui cara kerja atau penyelesaian dari materi tersebut.","Saat merasa belum paham, saya mencoba sendiri sampai mengetahui cara kerja atau penyelesaian dari materi tersebut. 2",IPK kumulatif saat ini,IPS semester terakhir,"Dalam satu semester terakhir, apakah Anda pernah mengulang mata kuliah?","Dalam satu semester terakhir, apakah Anda pernah mendapat nilai D/E?"
0,2026-03-17 16:11:14.219,Saya BERSEDIA mengikuti survei ini secara suka...,Arsitektur,6,Perempuan,20.0,Kost/asrama,Tidak,"Orang tua / keluarga, Beasiswa sebagian",1–2 jam,...,3.0,3.0,3.0,3.0,4.0,3.0,3.20,2.80,Tidak,Tidak
1,2026-03-17 16:23:55.853,Saya BERSEDIA mengikuti survei ini secara suka...,informatika,6,Perempuan,21.0,Bersama orang tua,Tidak,"Orang tua / keluarga, Beasiswa penuh",2–3 jam,...,3.0,3.0,3.0,2.0,3.0,4.0,3.07,3.91,Tidak,Tidak
2,2026-03-17 16:31:47.507,Saya BERSEDIA mengikuti survei ini secara suka...,Informatika,6,Laki-laki,20.0,Bersama orang tua,Tidak,Orang tua / keluarga,Kurang dari 1 jam,...,4.0,4.0,3.0,4.0,4.0,4.0,3.51,3.51,Tidak,Tidak


In [4]:
df_raw.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 89 columns):
 #   Column                                                                                                                         Non-Null Count  Dtype         
---  ------                                                                                                                         --------------  -----         
 0   Timestamp                                                                                                                      324 non-null    datetime64[us]
 1   Pernyataan Persetujuan                                                                                                         324 non-null    str           
 2    Program studi                                                                                                                 323 non-null    object        
 3    Semester saat ini                                                                    

In [5]:
df_raw.describe().round(2)

,Timestamp,Usia (tahun),Kualitas akses internet untuk belajar,Kehadiran perkuliahan saya selama satu semester terakhir secara umum,"Saya lebih mudah memahami materi jika disajikan dalam bentuk diagram, bagan, atau infografik.",Saya lebih mudah mengingat materi ketika dosen menggunakan slide yang jelas dan visual.,"Warna, simbol, atau penanda visual membantu saya memahami konsep.",Saya lebih suka membuat peta konsep atau mind map saat belajar.,Saya lebih cepat memahami penjelasan jika disertai ilustrasi atau video.,Saya lebih mudah memahami materi dengan mendengarkan penjelasan lisan dosen.,...,"Saat menemukan materi yang sulit, saya mencari penjelasan tertulis dari buku, modul, artikel, atau internet.","Saat menghafal konsep atau istilah, saya menuliskannya berulang kali agar lebih ingat.","Saat belajar di rumah, saya mengerjakan latihan soal, tugas, atau kuis untuk memahami materi.","Saat belajar sendiri, saya mencoba langsung langkah-langkah atau prosedur yang sedang dipelajari.","Saat mempelajari materi, saya menggunakan contoh kasus nyata agar lebih mudah memahami konsep.","Saat belajar di rumah, saya melakukan simulasi, praktik, eksperimen, atau demonstrasi kecil sesuai materi yang dipelajari.","Saat merasa belum paham, saya mencoba sendiri sampai mengetahui cara kerja atau penyelesaian dari materi tersebut.","Saat merasa belum paham, saya mencoba sendiri sampai mengetahui cara kerja atau penyelesaian dari materi tersebut. 2",IPK kumulatif saat ini,IPS semester terakhir
count,324,323.00,323.00,323.00,323.00,323.00,323.00,323.00,323.00,323.00,...,323.00,323.00,323.00,323.00,323.00,323.00,323.00,323.00,323.00,323.00
mean,2026-04-21 19:47:46.415814,20.93,4.04,4.58,3.73,4.32,4.36,3.48,4.31,3.75,...,4.02,3.77,3.87,3.93,3.99,3.73,4.02,4.05,10.91,10.32
min,2026-03-17 16:11:14.219000,18.00,1.00,2.00,1.00,1.00,2.00,1.00,2.00,1.00,...,2.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,2.00,0.00
25%,2026-04-14 22:12:47.149500,20.00,4.00,4.00,3.00,4.00,4.00,3.00,4.00,3.00,...,3.00,3.00,3.00,3.00,3.00,3.00,3.50,3.50,3.36,3.50
50%,2026-04-25 00:01:07.254500,21.00,4.00,5.00,4.00,4.00,4.00,3.00,4.00,4.00,...,4.00,4.00,4.00,4.00,4.00,4.00,4.00,4.00,3.60,3.70
75%,2026-04-28 21:40:19.310250,21.00,5.00,5.00,4.00,5.00,5.00,4.00,5.00,5.00,...,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,3.80,3.90
max,2026-05-02 14:11:52.871000,26.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,...,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,390.00,380.00
std,NaN,1.21,0.84,0.68,0.93,0.78,0.71,1.12,0.80,0.99,...,0.83,1.08,0.92,0.84,0.88,1.01,0.83,0.83,49.99,45.20


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(f"Jumlah data      : {df_raw.shape[0]}")
print(f"Jumlah kolom     : {df_raw.shape[1]}")
print()
dtype_summary = df_raw.dtypes.value_counts()
print("Tipe data:")
for dtype, count in dtype_summary.items():
    print(f"  {dtype}: {count} kolom")

Jumlah data      : 324
Jumlah kolom     : 89

Tipe data:
  float64: 77 kolom
  str: 9 kolom
  object: 2 kolom
  datetime64[us]: 1 kolom


: 

In [ ]:
col_ipk = '  IPK kumulatif saat ini  '
col_ips = ' IPS semester terakhir  '

ipk_valid = df_raw[df_raw[col_ipk].between(0, 4.0)][col_ipk].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(ipk_valid, bins=25, color='steelblue', edgecolor='white')
axes[0].axvline(ipk_valid.mean(), color='red', linestyle='--', label=f'Mean={ipk_valid.mean():.2f}')
axes[0].axvline(ipk_valid.median(), color='orange', linestyle='--', label=f'Median={ipk_valid.median():.2f}')
axes[0].set_title('Distribusi IPK Kumulatif (Valid)')
axes[0].set_xlabel('IPK'); axes[0].set_ylabel('Frekuensi')
axes[0].legend()

axes[1].boxplot(ipk_valid, vert=True)
axes[1].set_title('Boxplot IPK Kumulatif')
axes[1].set_ylabel('IPK')

ips_valid = df_raw[df_raw[col_ips].between(0, 4.0)][col_ips].dropna()
axes[2].hist(ips_valid, bins=25, color='coral', edgecolor='white')
axes[2].axvline(ips_valid.mean(), color='red', linestyle='--', label=f'Mean={ips_valid.mean():.2f}')
axes[2].set_title('Distribusi IPS Semester Terakhir')
axes[2].set_xlabel('IPS'); axes[2].set_ylabel('Frekuensi')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Statistik IPK (valid 0-4):")
print(ipk_valid.describe().round(3))

In [ ]:
col_gender = ' Jenis kelamin  '
col_kerja  = ' Apakah Anda bekerja sambil kuliah?  '
col_tinggal = ' Status tempat tinggal selama kuliah  '

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

gc = df_raw[col_gender].value_counts()
axes[0].pie(gc, labels=gc.index, autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('Set2', len(gc)))
axes[0].set_title('Jenis Kelamin')

wc = df_raw[col_kerja].value_counts()
wc.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', len(wc)))
axes[1].set_title('Status Bekerja Sambil Kuliah')
axes[1].set_xlabel(''); axes[1].set_ylabel('Jumlah')
axes[1].tick_params(axis='x', rotation=15)

tc = df_raw[col_tinggal].value_counts()
tc.plot(kind='barh', ax=axes[2], color=sns.color_palette('Set2', len(tc)))
axes[2].set_title('Status Tempat Tinggal')
axes[2].set_xlabel('Jumlah')

plt.tight_layout()
plt.show()

## 4. Data Checking

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Persen (%)': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
print(f"Kolom dengan missing value: {len(missing_df)}")
display(missing_df.head(15))

In [ ]:
obj_cols = df_raw.select_dtypes(include='object').columns
empty_str = {c: (df_raw[c].astype(str).str.strip() == '').sum()
             for c in obj_cols}
empty_str = {k: v for k, v in empty_str.items() if v > 0}
print("Kolom dengan empty string:", empty_str if empty_str else "Tidak ada")

print(f"Baris duplikat: {df_raw.duplicated().sum()}")

In [ ]:
col_ipk = '  IPK kumulatif saat ini  '

print("Statistik IPK (sebelum cleaning):")
print(df_raw[col_ipk].describe().round(3))
print()
print(f"IPK > 4.0  : {(df_raw[col_ipk] > 4.0).sum()} baris")
print(f"IPK < 0.0  : {(df_raw[col_ipk] < 0.0).sum()} baris")
print()
print("Nilai IPK tidak valid (> 4.0):")
display(df_raw[df_raw[col_ipk] > 4.0][[col_ipk, ' IPS semester terakhir  ']].reset_index())

In [ ]:
likert_cols = list(df_raw.columns[13:33]) + list(df_raw.columns[64:84])
outlier_summary = []
for c in likert_cols:
    vals = df_raw[c].dropna()
    Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((vals < Q1 - 1.5*IQR) | (vals > Q3 + 1.5*IQR)).sum()
    if n_out > 0:
        outlier_summary.append({'Kolom': c.strip()[:55], 'Outlier': n_out})

print(f"Kolom Likert dengan outlier (IQR): {len(outlier_summary)}")
if outlier_summary:
    display(pd.DataFrame(outlier_summary).head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(df_raw[col_ipk].dropna(), vert=True)
axes[0].set_title('Boxplot IPK (Sebelum Cleaning)')
axes[0].set_ylabel('IPK')

ipk_valid_check = df_raw[df_raw[col_ipk].between(0, 4.0)][col_ipk].dropna()
axes[1].boxplot(ipk_valid_check, vert=True)
axes[1].set_title('Boxplot IPK (Setelah Buang Outlier)')
axes[1].set_ylabel('IPK')

plt.tight_layout()
plt.show()

## 5. Data Preparation

In [ ]:
df = df_raw.copy()

drop_admin = [df.columns[0], df.columns[1]]
df.drop(columns=drop_admin, inplace=True)
print(f"[1] Hapus kolom admin -> {df.shape}")

In [ ]:
dup_cols = [c for c in df.columns if c.rstrip().endswith(' 2')]
print(f"Kolom duplikat: {len(dup_cols)}")
for c in dup_cols:
    print(f"  -> {c.strip()[:70]}")
df.drop(columns=dup_cols, inplace=True)
print(f"[2] Hapus kolom duplikat -> {df.shape}")

In [ ]:
col_ipk = '  IPK kumulatif saat ini  '
n_before = len(df)
df = df[df[col_ipk].between(0, 4.0) | df[col_ipk].isnull()].copy()
print(f"[3] Hapus IPK > 4.0 -> {n_before - len(df)} baris dihapus -> sisa {len(df)}")

In [ ]:
n_before = len(df)
df.dropna(thresh=int(len(df.columns) * 0.5), inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"[4] Hapus baris missing > 50% kolom -> {n_before - len(df)} baris dihapus -> sisa {len(df)}")

n_before = len(df)
df.dropna(subset=[col_ipk], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"[5] Hapus baris tanpa IPK -> {n_before - len(df)} baris -> sisa {len(df)}")

In [ ]:
def reverse_code(series):
    return 6 - series

col_E5 = [c for c in df.columns if 'Hambatan teknis' in c][0]
col_F5 = [c for c in df.columns if 'terdistraksi' in c][0]

df[col_E5] = reverse_code(df[col_E5])
df[col_F5] = reverse_code(df[col_F5])
print(f"[6] Reverse coding selesai: E5 dan F5 dibalik.")

col_prodi = ' Program studi  '
df[col_prodi] = df[col_prodi].astype(str).str.strip().str.title()
print(f"[7] Program studi distandarkan -> {df[col_prodi].nunique()} nilai unik")

print(f"\nDataset bersih: {df.shape}")

## 6. Feature Engineering — Membuat LearningStyle
LearningStyle dibuat ulang menggunakan aturan yang sama dengan Notebook 1:
- Hitung rata-rata 10 item per dimensi (5 attitude + 5 konfirmasi)
- Label = dimensi dengan skor tertinggi


In [ ]:
visual_att    = [c for c in df.columns if any(k in c for k in [
    'diagram, bagan, atau infografik',
    'slide yang jelas dan visual',
    'Warna, simbol, atau penanda visual',
    'peta konsep atau mind map saat belajar',
    'ilustrasi atau video'])]

auditory_att  = [c for c in df.columns if any(k in c for k in [
    'mendengarkan penjelasan lisan dosen',
    'Diskusi kelas atau diskusi kelompok',
    'membacanya keras-keras atau menjelaskannya secara verbal',
    'mendengar contoh yang dijelaskan secara lisan',
    'rekaman audio atau penjelasan verbal membantu'])]

readwrite_att = [c for c in df.columns if any(k in c for k in [
    'buku, modul, atau artikel tertulis',
    'mencatat ulang poin-poin penting',
    'rangkuman tertulis setelah perkuliahan',
    'instruksi dalam bentuk tulisan daripada penjelasan lisan',
    'istilah atau konsep ketika menuliskannya'])]

kinesthetic_att = [c for c in df.columns if any(k in c for k in [
    'langsung mempraktikkannya',
    'Contoh kasus nyata membuat saya lebih cepat',
    'simulasi, eksperimen, atau demonstrasi',
    'mencoba sendiri langkah-langkahnya',
    'praktik daripada hanya membaca teori'])]

visual_conf    = [c for c in df.columns if any(k in c for k in [
    'slide, diagram, bagan, atau infografik untuk memahami',
    'mencari video pembelajaran untuk membantu',
    'highlight, warna, simbol, atau tanda khusus',
    'mind map, peta konsep, atau skema ringkas',
    'gambar, ilustrasi, tabel, atau tampilan visual lain'])]

auditory_conf  = [c for c in df.columns if any(k in c for k in [
    'mendengarkan ulang rekaman penjelasan atau audio',
    'suara keras agar lebih mudah memahami isi',
    'menjelaskan materi dengan lisan kepada diri sendiri',
    'berdiskusi dengan teman, baik secara langsung',
    'penjelasan berbentuk suara atau video yang menjelaskan materi secara verbal'])]

readwrite_conf = [c for c in df.columns if any(k in c for k in [
    'membaca ulang catatan kuliah, modul',
    'menulis ulang poin-poin penting dari materi',
    'rangkuman tertulis dengan bahasa saya sendiri',
    'penjelasan tertulis dari buku, modul, artikel, atau internet',
    'menuliskannya berulang kali agar lebih ingat'])]

kinesthetic_conf = [c for c in df.columns if any(k in c for k in [
    'latihan soal, tugas, atau kuis untuk memahami',
    'mencoba langsung langkah-langkah atau prosedur',
    'contoh kasus nyata agar lebih mudah memahami konsep',
    'simulasi, praktik, eksperimen, atau demonstrasi kecil',
    'mencoba sendiri sampai mengetahui cara kerja'])]

visual_all      = visual_att    + visual_conf
auditory_all    = auditory_att  + auditory_conf
readwrite_all   = readwrite_att + readwrite_conf
kinesthetic_all = kinesthetic_att + kinesthetic_conf

print("Jumlah item per dimensi VARK:")
for name, cols in [('Visual', visual_all), ('Auditory', auditory_all),
                    ('ReadWrite', readwrite_all), ('Kinesthetic', kinesthetic_all)]:
    print(f"  {name}: {len(cols)} item")

In [ ]:
df['score_Visual']      = df[visual_all].mean(axis=1)
df['score_Auditory']    = df[auditory_all].mean(axis=1)
df['score_ReadWrite']   = df[readwrite_all].mean(axis=1)
df['score_Kinesthetic'] = df[kinesthetic_all].mean(axis=1)

score_cols = ['score_Visual', 'score_Auditory', 'score_ReadWrite', 'score_Kinesthetic']
df['LearningStyle'] = df[score_cols].idxmax(axis=1).map({
    'score_Visual'     : 'Visual',
    'score_Auditory'   : 'Auditory',
    'score_ReadWrite'  : 'ReadWrite',
    'score_Kinesthetic': 'Kinesthetic'
})

print("Distribusi LearningStyle:")
print(df['LearningStyle'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ls_counts = df['LearningStyle'].value_counts()
colors = sns.color_palette('Set2', 4)
bars = axes[0].bar(ls_counts.index, ls_counts.values, color=colors)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                  str(int(bar.get_height())), ha='center', fontweight='bold')
axes[0].set_title('Distribusi LearningStyle')
axes[0].set_xlabel('Gaya Belajar'); axes[0].set_ylabel('Jumlah')

df.boxplot(column=col_ipk, by='LearningStyle', ax=axes[1],
           boxprops=dict(color='steelblue'))
axes[1].set_title('Distribusi IPK per LearningStyle')
axes[1].set_xlabel('LearningStyle'); axes[1].set_ylabel('IPK')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 7. Menentukan Fitur dan Target
Target: `IPK kumulatif saat ini`. Fitur mencakup LearningStyle, demografi, kebiasaan belajar (D), lingkungan (E), teknologi (F), dan IPS semester terakhir. Item VARK dan skor turunannya tidak digunakan sebagai fitur (data leakage).

In [ ]:
CAT_FEATURES = [
    ' Jenis kelamin  ',
    ' Status tempat tinggal selama kuliah  ',
    ' Apakah Anda bekerja sambil kuliah?  ',
    ' Rata-rata waktu belajar mandiri per hari di luar jam kuliah  ',
    ' Perangkat utama yang digunakan untuk belajar  ',
    'Dalam satu semester terakhir, apakah Anda pernah mengulang mata kuliah?  ',
    'Dalam satu semester terakhir, apakah Anda pernah mendapat nilai D/E? ',
    'LearningStyle',
]

NUM_BASE = [
    ' Usia  (tahun)',
    ' Kualitas akses internet untuk belajar  ',
    ' Kehadiran perkuliahan saya selama satu semester terakhir secara umum  ',
    ' IPS semester terakhir  ',
]

df[' Semester saat ini  '] = pd.to_numeric(df[' Semester saat ini  '], errors='coerce')
NUM_BASE.append(' Semester saat ini  ')

D_COLS = [c for c in df.columns if any(k in c for k in [
    'jadwal belajar yang cukup teratur',
    'membagi waktu belajar jauh sebelum ujian',
    'menentukan prioritas antara kuliah',
    'jarang menunda mengerjakan tugas',
    'konsisten mengalokasikan waktu khusus',
    'meninjau kembali materi kuliah setelah',
    'mencari sumber tambahan ketika ada materi',
    'membandingkan berbagai sumber untuk',
    'membuat ringkasan, poin penting, atau catatan pribadi',
    'mengecek sendiri apakah saya benar-benar',
    'memiliki target akademik yang jelas',
    'tetap berusaha memahami materi walaupun terasa sulit',
    'terdorong untuk belajar bukan hanya demi nilai',
    'merasa bertanggung jawab terhadap hasil',
    'berusaha memperbaiki strategi belajar ketika',
    'aktif bertanya atau berdiskusi ketika ada materi',
    'mengerjakan tugas kuliah dengan sungguh',
    'mengikuti perkuliahan dengan fokus',
    'berusaha hadir tepat waktu dalam perkuliahan',
    'memanfaatkan umpan balik dari dosen'])]

E_COLS = [c for c in df.columns if any(k in c for k in [
    'tempat belajar yang cukup nyaman',
    'lingkungan tempat saya belajar mendukung',
    'akses perangkat yang memadai',
    'dukungan sosial yang cukup',
    'Hambatan teknis seperti internet'])]

F_COLS = [c for c in df.columns if any(k in c for k in [
    'platform digital untuk memahami',
    'Video pembelajaran atau tutorial online',
    'aplikasi atau tools digital untuk mencatat',
    'belajar secara efektif melalui media pembelajaran online',
    'terdistraksi oleh penggunaan perangkat digital'])]

NUM_FEATURES = NUM_BASE + D_COLS + E_COLS + F_COLS
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

print(f"Fitur kategorikal : {len(CAT_FEATURES)}")
print(f"Fitur numerikal   : {len(NUM_FEATURES)}")
print(f"  Numerik base    : {len(NUM_BASE)}")
print(f"  Kolom D         : {len(D_COLS)}")
print(f"  Kolom E         : {len(E_COLS)}")
print(f"  Kolom F         : {len(F_COLS)}")
print(f"Total fitur       : {len(ALL_FEATURES)}")

In [ ]:
X = df[ALL_FEATURES].copy()
y = df[col_ipk].copy()

print(f"Shape X : {X.shape}")
print(f"Shape y : {y.shape}")
print(f"\nTarget (IPK) - statistik:")
print(y.describe().round(3))

## 8. Korelasi dan Analisis Fitur

In [ ]:
num_data = df[NUM_FEATURES + [col_ipk]].copy()
corr_with_ipk = num_data.corr()[col_ipk].drop(col_ipk).sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 7))
colors = ['steelblue' if v >= 0 else 'coral' for v in corr_with_ipk.values]
corr_with_ipk.head(20).plot(kind='barh', color=colors[:20])
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Top 20 Korelasi Fitur Numerik dengan IPK')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()

print("Top 10 fitur berkorelasi dengan IPK:")
print(corr_with_ipk.head(10).round(3).to_string())

In [ ]:
top_corr_cols = corr_with_ipk.head(10).index.tolist() + [col_ipk]
corr_matrix = df[top_corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Heatmap Korelasi - Top 10 Fitur + IPK')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Train Test Split

In [ ]:
from sklearn.linear_model import Ridge as RidgeCheck

results_split = []
for test_sz, label in [(0.20, '80:20'), (0.25, '75:25')]:
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=test_sz, random_state=RANDOM_STATE)

    num_tr = Xtr[NUM_FEATURES].apply(pd.to_numeric, errors='coerce')
    num_te = Xte[NUM_FEATURES].apply(pd.to_numeric, errors='coerce')
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    Xtr_s = scl.fit_transform(imp.fit_transform(num_tr))
    Xte_s = scl.transform(imp.transform(num_te))

    model_check = RidgeCheck()
    model_check.fit(Xtr_s, ytr)
    y_pred_check = model_check.predict(Xte_s)
    rmse = np.sqrt(mean_squared_error(yte, y_pred_check))
    r2   = r2_score(yte, y_pred_check)
    results_split.append({'Split': label, 'Train': len(Xtr), 'Test': len(Xte),
                           'RMSE (Ridge)': round(rmse, 4), 'R2 (Ridge)': round(r2, 4)})

split_df = pd.DataFrame(results_split)
display(split_df)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  |  y_test: {y_test.shape}")
print(f"\nIPK train - mean: {y_train.mean():.3f}, std: {y_train.std():.3f}")
print(f"IPK test  - mean: {y_test.mean():.3f}, std: {y_test.std():.3f}")

## 10. Preprocessing Pipeline

In [ ]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, NUM_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES)
], remainder='drop')

X_temp = preprocessor.fit_transform(X_train, y_train)
N_FEAT = X_temp.shape[1]
print(f"Jumlah fitur setelah preprocessing: {N_FEAT}")

## 11. Feature Selection

In [ ]:
from sklearn.feature_selection import f_regression, mutual_info_regression

X_prep = preprocessor.fit_transform(X_train)

f_scores, _   = f_regression(X_prep, y_train)
mi_scores     = mutual_info_regression(X_prep, y_train, random_state=RANDOM_STATE)

fs_df = pd.DataFrame({'f_regression': f_scores, 'mutual_info': mi_scores})
fs_df['rank_f']  = fs_df['f_regression'].rank(ascending=False)
fs_df['rank_mi'] = fs_df['mutual_info'].rank(ascending=False)

print("Statistik skor feature selection:")
print(fs_df[['f_regression', 'mutual_info']].describe().round(3))

## 12. Model Regresi
Lima model dibandingkan, masing-masing dalam Pipeline lengkap:
`Preprocessing → Feature Selection → Regressor`


In [ ]:
pipe_lr = Pipeline([
    ('prep',     preprocessor),
    ('selector', SelectKBest(score_func=f_regression)),
    ('reg',      LinearRegression())
])

pipe_ridge = Pipeline([
    ('prep',     preprocessor),
    ('selector', SelectKBest(score_func=f_regression)),
    ('reg',      Ridge(random_state=RANDOM_STATE))
])

pipe_rf = Pipeline([
    ('prep',     preprocessor),
    ('selector', SelectKBest(score_func=mutual_info_regression)),
    ('reg',      RandomForestRegressor(random_state=RANDOM_STATE))
])

pipe_svr = Pipeline([
    ('prep',     preprocessor),
    ('selector', SelectPercentile(score_func=f_regression)),
    ('reg',      SVR())
])

pipe_gb = Pipeline([
    ('prep',     preprocessor),
    ('selector', SelectKBest(score_func=mutual_info_regression)),
    ('reg',      GradientBoostingRegressor(random_state=RANDOM_STATE))
])

pipe_enet = Pipeline([
    ('prep',     preprocessor),
    ('selector', SelectKBest(score_func=f_regression)),
    ('reg',      ElasticNet(random_state=RANDOM_STATE, max_iter=10000))
])

pipe_et = Pipeline([
    ('prep',     preprocessor),
    ('selector', SelectKBest(score_func=mutual_info_regression)),
    ('reg',      ExtraTreesRegressor(random_state=RANDOM_STATE))
])

## 13. Hyperparameter Tuning (GridSearchCV)

In [ ]:
cv_kfold = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
k_options = [10, 15, 20, 25]

param_lr = {'selector__k': k_options}

gs_lr = GridSearchCV(pipe_lr, param_lr, cv=cv_kfold,
                     scoring='r2', n_jobs=-1)
gs_lr.fit(X_train, y_train)
print(f"Linear Regression — Best Params : {gs_lr.best_params_}")
print(f"Linear Regression — Best CV R2  : {gs_lr.best_score_:.4f}")

In [ ]:
param_ridge = {
    'selector__k': k_options,
    'reg__alpha'  : [0.1, 1.0, 10.0, 100.0],
}

gs_ridge = GridSearchCV(pipe_ridge, param_ridge, cv=cv_kfold,
                        scoring='r2', n_jobs=-1)
gs_ridge.fit(X_train, y_train)
print(f"Ridge — Best Params : {gs_ridge.best_params_}")
print(f"Ridge — Best CV R2  : {gs_ridge.best_score_:.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_rf = {
    'selector__k'           : k_options,
    'reg__n_estimators'     : [100, 200, 300],
    'reg__max_depth'        : [None, 5, 10],
    'reg__min_samples_split': [2, 5],
    'reg__min_samples_leaf' : [1, 2],
}

gs_rf = RandomizedSearchCV(pipe_rf, param_rf, n_iter=30,
                           cv=cv_kfold, scoring='r2',
                           random_state=RANDOM_STATE, n_jobs=-1)
gs_rf.fit(X_train, y_train)
print(f"Random Forest — Best Params : {gs_rf.best_params_}")
print(f"Random Forest — Best CV R2  : {gs_rf.best_score_:.4f}")

In [ ]:
param_svr = {
    'selector__percentile': [25, 50, 75, 100],
    'reg__C'              : [1, 10, 100],
    'reg__epsilon'        : [0.05, 0.1],
    'reg__kernel'         : ['rbf', 'linear'],
}

gs_svr = GridSearchCV(pipe_svr, param_svr, cv=cv_kfold,
                      scoring='r2', n_jobs=-1)
gs_svr.fit(X_train, y_train)
print(f"SVR — Best Params : {gs_svr.best_params_}")
print(f"SVR — Best CV R2  : {gs_svr.best_score_:.4f}")

In [ ]:
param_gb = {
    'selector__k'       : k_options,
    'reg__n_estimators' : [100, 200, 300],
    'reg__learning_rate': [0.05, 0.1, 0.2],
    'reg__max_depth'    : [3, 5],
}

gs_gb = RandomizedSearchCV(pipe_gb, param_gb, n_iter=20,
                           cv=cv_kfold, scoring='r2',
                           random_state=RANDOM_STATE, n_jobs=-1)
gs_gb.fit(X_train, y_train)
print(f"Gradient Boosting — Best Params : {gs_gb.best_params_}")
print(f"Gradient Boosting — Best CV R2  : {gs_gb.best_score_:.4f}")

In [ ]:
param_enet = {
    'selector__k'   : k_options,
    'reg__alpha'    : [0.01, 0.1, 1.0],
    'reg__l1_ratio' : [0.2, 0.5, 0.8],
}

gs_enet = GridSearchCV(pipe_enet, param_enet, cv=cv_kfold,
                       scoring='r2', n_jobs=-1)
gs_enet.fit(X_train, y_train)
print(f"ElasticNet — Best Params : {gs_enet.best_params_}")
print(f"ElasticNet — Best CV R2  : {gs_enet.best_score_:.4f}")

In [ ]:
param_et = {
    'selector__k'           : k_options,
    'reg__n_estimators'     : [100, 200, 300],
    'reg__max_depth'        : [None, 5, 10],
    'reg__min_samples_split': [2, 5],
}

gs_et = RandomizedSearchCV(pipe_et, param_et, n_iter=25,
                           cv=cv_kfold, scoring='r2',
                           random_state=RANDOM_STATE, n_jobs=-1)
gs_et.fit(X_train, y_train)
print(f"Extra Trees — Best Params : {gs_et.best_params_}")
print(f"Extra Trees — Best CV R2  : {gs_et.best_score_:.4f}")

In [ ]:
cv_summary = {
    'Linear Regression' : gs_lr.best_score_,
    'Ridge'             : gs_ridge.best_score_,
    'Random Forest'     : gs_rf.best_score_,
    'SVR'               : gs_svr.best_score_,
    'Gradient Boosting' : gs_gb.best_score_,
    'ElasticNet'        : gs_enet.best_score_,
    'Extra Trees'       : gs_et.best_score_,
}
print("Ringkasan Best CV R2:")
for m, s in sorted(cv_summary.items(), key=lambda x: -x[1]):
    print(f"  {m:<22}: {s:.4f}")

## 14. Cross Validation (KFold)

In [ ]:
best_models = {
    'Linear Regression' : gs_lr.best_estimator_,
    'Ridge'             : gs_ridge.best_estimator_,
    'Random Forest'     : gs_rf.best_estimator_,
    'SVR'               : gs_svr.best_estimator_,
    'Gradient Boosting' : gs_gb.best_estimator_,
    'ElasticNet'        : gs_enet.best_estimator_,
    'Extra Trees'       : gs_et.best_estimator_,
}

cv_table = []
for name, model in best_models.items():
    cv_r2   = cross_val_score(model, X_train, y_train, cv=cv_kfold,
                              scoring='r2', n_jobs=-1)
    cv_rmse = np.sqrt(-cross_val_score(model, X_train, y_train, cv=cv_kfold,
                                       scoring='neg_mean_squared_error', n_jobs=-1))
    cv_table.append({
        'Model'       : name,
        'CV R2 Mean'  : cv_r2.mean(),
        'CV R2 Std'   : cv_r2.std(),
        'CV RMSE Mean': cv_rmse.mean(),
        'CV RMSE Std' : cv_rmse.std(),
    })

cv_df = pd.DataFrame(cv_table).sort_values('CV R2 Mean', ascending=False).reset_index(drop=True)
for col in ['CV R2 Mean','CV R2 Std','CV RMSE Mean','CV RMSE Std']:
    cv_df[col] = cv_df[col].round(4)
display(cv_df)

## 15. Evaluasi Model pada Test Set

In [ ]:
eval_results = []
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    eval_results.append({
        'Model': name,
        'R2'   : r2_score(y_test, y_pred),
        'MAE'  : mean_absolute_error(y_test, y_pred),
        'MSE'  : mean_squared_error(y_test, y_pred),
        'RMSE' : np.sqrt(mean_squared_error(y_test, y_pred)),
    })

eval_df = pd.DataFrame(eval_results).sort_values('R2', ascending=False).reset_index(drop=True)
for col in ['R2','MAE','MSE','RMSE']:
    eval_df[col] = eval_df[col].round(4)
display(eval_df)

## 16. Visualisasi Wajib

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y, bins=25, color='steelblue', edgecolor='white')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean={y.mean():.2f}')
axes[0].axvline(y.median(), color='orange', linestyle='--', label=f'Median={y.median():.2f}')
axes[0].set_title('Distribusi IPK (Dataset Bersih)')
axes[0].set_xlabel('IPK'); axes[0].set_ylabel('Frekuensi')
axes[0].legend()

import scipy.stats as stats
stats.probplot(y, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot IPK')

plt.tight_layout()
plt.show()

In [ ]:
pairplot_cols = corr_with_ipk.head(4).index.tolist() + [col_ipk]
pairplot_data = df[pairplot_cols].dropna()
pairplot_data.columns = [c.strip()[:30] for c in pairplot_data.columns]

pp = sns.pairplot(pairplot_data, diag_kind='kde',
                  plot_kws={'alpha': 0.5, 'color': 'steelblue'})
pp.fig.suptitle('Pairplot - Top 4 Fitur Numerik + IPK', y=1.02)
plt.show()

In [ ]:
n_models = len(best_models)
n_cols = 4
n_rows = math.ceil(n_models / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, (name, model) in enumerate(best_models.items()):
    y_pred = model.predict(X_test)
    axes[idx].scatter(y_test, y_pred, alpha=0.6, color='steelblue', s=30)
    lims = [min(y_test.min(), y_pred.min()) - 0.1,
            max(y_test.max(), y_pred.max()) + 0.1]
    axes[idx].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect')
    axes[idx].set_xlabel('Actual IPK')
    axes[idx].set_ylabel('Predicted IPK')
    r2_val = r2_score(y_test, y_pred)
    axes[idx].set_title(f'{name}\nR2={r2_val:.3f}')
    axes[idx].legend(fontsize=8)

for j in range(n_models, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
n_models = len(best_models)
n_cols = 4
n_rows = math.ceil(n_models / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, (name, model) in enumerate(best_models.items()):
    y_pred   = model.predict(X_test)
    residuals = y_test - y_pred
    axes[idx].scatter(y_pred, residuals, alpha=0.6, color='coral', s=30)
    axes[idx].axhline(y=0, color='black', linestyle='--', linewidth=1)
    axes[idx].set_xlabel('Predicted IPK')
    axes[idx].set_ylabel('Residual')
    axes[idx].set_title(f'Residual - {name}')

for j in range(n_models, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
best_name = eval_df.iloc[0]['Model']
best_model = best_models[best_name]

y_pred_best = best_model.predict(X_test)

sample20 = pd.DataFrame({
    'No'        : range(1, 21),
    'Actual IPK': y_test.iloc[:20].values.round(2),
    'Predicted' : y_pred_best[:20].round(2),
    'Error'     : (y_test.iloc[:20].values - y_pred_best[:20]).round(3),
    'Abs Error' : np.abs(y_test.iloc[:20].values - y_pred_best[:20]).round(3)
}).set_index('No')

print(f"20 Data Test Pertama - Model: {best_name}")
display(sample20)
print(f"\nMAE 20 sampel: {sample20['Abs Error'].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

x20 = np.arange(1, 21)
axes[0].plot(x20, sample20['Actual IPK'], 'o-', color='steelblue', label='Actual', linewidth=2)
axes[0].plot(x20, sample20['Predicted'], 's--', color='coral', label='Predicted', linewidth=2)
axes[0].set_title(f'Actual vs Predicted IPK - 20 Sampel Test Pertama ({best_name})')
axes[0].set_xlabel('Nomor Sampel'); axes[0].set_ylabel('IPK')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].bar(x20, sample20['Error'], color=['steelblue' if e >= 0 else 'coral'
                                             for e in sample20['Error']])
axes[1].axhline(y=0, color='black', linewidth=1)
axes[1].set_title('Error (Actual - Predicted) per Sampel')
axes[1].set_xlabel('Nomor Sampel'); axes[1].set_ylabel('Error')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 17. Perbandingan Model

In [ ]:
print("Tabel Perbandingan Model (diurutkan R2 terbaik):")
display(eval_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = sns.color_palette('Set2', len(eval_df))

bars = axes[0].barh(eval_df['Model'], eval_df['R2'], color=colors)
for bar, val in zip(bars, eval_df['R2']):
    axes[0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                  f'{val:.4f}', va='center', fontsize=9)
axes[0].set_title('R2 Score per Model')
axes[0].set_xlabel('R2')
axes[0].set_xlim(0, max(eval_df['R2']) + 0.1)

eval_sorted_rmse = eval_df.sort_values('RMSE')
bars2 = axes[1].barh(eval_sorted_rmse['Model'], eval_sorted_rmse['RMSE'], color=colors)
for bar, val in zip(bars2, eval_sorted_rmse['RMSE']):
    axes[1].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                  f'{val:.4f}', va='center', fontsize=9)
axes[1].set_title('RMSE per Model (lebih kecil = lebih baik)')
axes[1].set_xlabel('RMSE')

plt.tight_layout()
plt.show()

## 18. Analisis Feature Importance

In [ ]:
rf_best = gs_rf.best_estimator_

ohe_rf = rf_best.named_steps['prep'].named_transformers_['cat'].named_steps['encoder']
cat_fn = list(ohe_rf.get_feature_names_out(CAT_FEATURES))
all_fn = NUM_FEATURES + cat_fn

sel_rf   = rf_best.named_steps['selector']
sel_mask = sel_rf.get_support()
sel_fn   = [all_fn[i] for i, s in enumerate(sel_mask) if s]

fi_values = rf_best.named_steps['reg'].feature_importances_
fi_df = pd.DataFrame({
    'Feature'   : [f.strip()[:60] for f in sel_fn],
    'Importance': fi_values
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='viridis')
plt.title('Top 20 Feature Importance - Random Forest Regressor')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
gb_best = gs_gb.best_estimator_

ohe_gb = gb_best.named_steps['prep'].named_transformers_['cat'].named_steps['encoder']
cat_fn_gb = list(ohe_gb.get_feature_names_out(CAT_FEATURES))
all_fn_gb = NUM_FEATURES + cat_fn_gb

sel_gb   = gb_best.named_steps['selector']
sel_mask_gb = sel_gb.get_support()
sel_fn_gb = [all_fn_gb[i] for i, s in enumerate(sel_mask_gb) if s]

fi_gb = pd.DataFrame({
    'Feature'   : [f.strip()[:60] for f in sel_fn_gb],
    'Importance': gb_best.named_steps['reg'].feature_importances_
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=fi_gb, x='Importance', y='Feature', palette='magma')
plt.title('Top 20 Feature Importance - Gradient Boosting Regressor')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 19. Pemilihan Model Terbaik

In [ ]:
display(eval_df)
display(cv_df[['Model','CV R2 Mean','CV R2 Std','CV RMSE Mean','CV RMSE Std']])

best_name = eval_df.iloc[0]['Model']
best_r2   = eval_df.iloc[0]['R2']
best_rmse = eval_df.iloc[0]['RMSE']

print(f"\nModel terbaik: {best_name}")
print(f"R2   (Test Set): {best_r2:.4f}")
print(f"RMSE (Test Set): {best_rmse:.4f}")

## 20. Export Model

In [ ]:
best_estimator_map = {
    'Linear Regression' : gs_lr.best_estimator_,
    'Ridge'             : gs_ridge.best_estimator_,
    'Random Forest'     : gs_rf.best_estimator_,
    'SVR'               : gs_svr.best_estimator_,
    'Gradient Boosting' : gs_gb.best_estimator_,
    'ElasticNet'        : gs_enet.best_estimator_,
    'Extra Trees'       : gs_et.best_estimator_,
}

best_pipeline = best_estimator_map[best_name]

output_path = 'BestModel_Regresi_Alg_William_Luvianus_Selly_Monica.pkl'
joblib.dump(best_pipeline, output_path)
print(f"Model terbaik ({best_name}) disimpan ke: {output_path}")
print(f"Disimpan dengan scikit-learn versi: {sklearn.__version__}")

loaded = joblib.load(output_path)
sample_pred = loaded.predict(X_test.iloc[:5])
print(f"\nVerifikasi prediksi 5 sampel:")
print(f"  Prediksi : {sample_pred.round(3).tolist()}")
print(f"  Aktual   : {y_test.iloc[:5].round(3).tolist()}")

---
## Kesimpulan

Model terbaik diekspor ke `BestModel_Regresi_Alg_William_Luvianus_Selly_Monica.pkl` untuk digunakan di aplikasi Streamlit.